# GR1D Viz: choose a plot, then edit its settings

This notebook uses the same public functions as a Python script. Run the setup first,
then use the section for the plot you need. Each example states its input, axes,
and return value. Install with `python -m pip install -e ".[notebook,eos]"` from the
package directory. The data stay on your machine.

| Function | What it shows | Returns |
|---|---|---|
| `quicklook(runs)` | Central diagnostics and shock radius; compare progenitors | `fig, axes` |
| `series(runs, variable)` | Selected scalar diagnostic versus time | `fig, ax` |
| `profiles(runs, panels)` | Radial profiles at selected times | `fig, axes` |
| `animate_profiles(run, panels)` | The radial profiles evolving in time | `ani` |
| `star(run, variable)` | One radial variable repeated around a polar cross-section | `fig, ax` |
| `star(..., animate=True)` | The cross-section evolving in time | `ani` |

To inspect the full signature and docstring, run `help(profiles)` (or any other function).


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from gr1d_viz import (Run, Panel, quicklook, series, profiles,
                       animate_profiles, star, save_animation, configure_style)
from gr1d_viz.constants import CM_TO_KM, VELOCITY_TO_C, GRAMS_TO_SOLAR_MASS

# One output folder per run/progenitor. Add more entries to compare them.
# Use a core-collapse run with mass_grav.xg, temperature_c_t.dat and tbounce.dat.
RUN_FOLDERS = {"s24": "/path/to/GR1D/output/s24"}
USE_LATEX = False                 # True requires a working external TeX installation
USE_SEABORN = True               # Matplotlib's bundled seaborn paper style
RUN_ANIMATIONS = False           # Enable after choosing a short time window

configure_style(seaborn=USE_SEABORN, latex=USE_LATEX)
runs = [Run(folder, label=label, cache_dir=".gr1d-cache")
        for label, folder in RUN_FOLDERS.items()]
run = runs[0]                   # Animations and EOS examples use this selected run


## 1. Inspect the available files

`available()` separates radius profiles, scalar series, metadata, timers, spectra,
mass-coordinate snapshots, special profiles, binary files and unknown files.
`tbounce.dat` is metadata. It must not be plotted as a scalar time series.
Unknown `.dat` files require explicit column choices after you inspect their format.


In [ ]:
display(run.available())
display(run.inspect_files())
print("Bounce time:", run.bounce_time, "s")
# run.bounce_metadata contains the other raw entries in tbounce.dat.


## 2. Quicklook: compare progenitors at a glance

**Input:** one Run or a list of Runs. **Horizontal axis:** seconds.
**Panels:** central density, temperature, electron fraction, shock radius, where available.
**Return:** a Figure and a list of primary axes. A run has the same color in every panel.
Missing diagnostics are skipped with a warning; each run uses its own bounce time.


In [ ]:
fig, axes = quicklook(runs, postbounce=True)
# fig.savefig("quicklook.png", dpi=200)
plt.show()


## 3. Series: one diagnostic versus time

**Input:** a `.dat` stem and optional columns. **Return:** `fig, ax`.
This example plots central temperature. For a multi-column file such as
`M1_flux_lum.dat`, select `value_column=1`, `2`, or `3` explicitly.
For `timers.dat`, `value_column=2` selects the c2p timer; the reader knows that
simulation time is column 1. Leakage diagnostics already written in post-bounce
time are converted correctly rather than subtracting bounce twice.


In [ ]:
fig, ax = series(runs, "temperature_c_t", postbounce=True,
                 ylabel="Central temperature (MeV)")
# ax.set_xlim(0.0, 0.5)
# fig.savefig("temperature.png", dpi=200)
plt.show()


## 4. Profiles: radial snapshots

**Horizontal axis:** radius in km. **Return:** `fig, axes` (primary axes only).
`times=None` uses each variable's last frame. A requested time snaps to the nearest
recorded frame and the legend reports that actual time. Each variable has its own
color, including on twin axes; line styles distinguish times/runs.

The default below requires `mass_grav.xg`. For a full output folder,
replace `panels` with density, temperature or velocity examples shown below.


In [ ]:
panels = [Panel("mass_grav", factor=GRAMS_TO_SOLAR_MASS,
                ylabel=r"Gravitational mass ($M_\odot$)")]
# Alternative, requiring rho.xg, temperature.xg and v.xg:
# panels = [Panel("rho", yscale="log", ylim=(1e8, 1e15)),
#           Panel("v", factor=VELOCITY_TO_C, ylabel="v/c", twin=True, hlines=(0,)),
#           Panel("temperature", ylim=(0, 75))]

SNAPSHOT_TIMES = None             # e.g. [0.0, 0.1] with postbounce=True
fig, axes = profiles(runs, panels, times=SNAPSHOT_TIMES,
                     postbounce=True, xlim=(0, 100), title="Radial profiles")
plt.show()


### Edit an existing figure without reading data again
`Panel` is a frozen settings object: create a new one or use `dataclasses.replace`
for a new plot. To change the figure already drawn, edit the returned axes:


In [ ]:
axes[0].set_xlim(0, 30)
fig.canvas.draw_idle()
display(fig)


## 5. Animate radial panels

**Input:** one Run and the same panel list. **Return:** a Matplotlib animation.
`window` is in seconds; `stride` skips frames; `interval` is playback delay in ms.
The example chooses a short simulation-time segment near bounce and limits the
number of frames. Enable `RUN_ANIMATIONS` in setup to render it in the notebook.


In [ ]:
p = run.profile(panels[0].variable)
WINDOW = (max(p.times[0], run.bounce_time - 0.02),
          min(p.times[-1], run.bounce_time + 0.02))
STRIDE = max(1, int(sum((p.times >= WINDOW[0]) & (p.times <= WINDOW[1])) / 30))
if RUN_ANIMATIONS:
    ani = animate_profiles(run, panels, window=WINDOW, stride=STRIDE,
                           xlim=(0, 100), interval=50)
    display(HTML(ani.to_jshtml()))
    # save_animation(ani, "panels.gif", fps=20)
    plt.close("all")
else:
    print("Set RUN_ANIMATIONS=True to render the selected short window.")


## 6. Polar cross-section

**Input:** one radial variable. **Return:** `fig, ax`, or `ani` with `animate=True`.
`max_radius` is km; `vmin/vmax` fix the color scale; `log` controls color scaling.
`radius_label_color` changes the radial tick-label color.
The example colors by enclosed gravitational mass in grams. For a density view,
use `variable="rho"`, `log=True`, and suitable density color limits.
This is a spherical replication of 1D data, not a map of angular structure.


In [ ]:
STAR_VARIABLE = "mass_grav"
fig, ax = star(run, STAR_VARIABLE, max_radius=100, log=False,
               radius_label_color="white", cmap="inferno",
               label="Enclosed gravitational mass (g)")
plt.show()


In [ ]:
if RUN_ANIMATIONS:
    star_ani = star(run, STAR_VARIABLE, animate=True, window=WINDOW,
                    stride=STRIDE, max_radius=100, log=False,
                    radius_label_color="cyan")
    display(HTML(star_ani.to_jshtml()))
    # save_animation(star_ani, "star.gif", fps=20)
    plt.close("all")


## 7. EOS lookup: what the last cell shows

The final plot is **quark fraction versus radius at one simulation time**.
For each radial zone, the code takes that zone's density, temperature and electron
fraction, looks up `Xu`, `Xd`, and `Xs` in the selected EOS, and adds them. The
result is a derived radial profile, registered on the Run as `quark_fraction`.

The horizontal axis is **radius (km)**. The left axis is **density (g/cm³)**;
the right axis is **Xu + Xd + Xs**, dimensionless. It is not an EOS phase diagram.
The selected table must use the same fraction convention as your model; the code
sums the supplied datasets without redefining them. Outside the EOS domain,
results are NaN and appear as gaps.

You may use a thermodynamics file and a separate composition file. Each file keeps its own grid. Choose `source=` when a dataset
exists in multiple files. To compare different EOS models, create separate EOS
objects or use named sources explicitly; no tables are silently merged.


In [ ]:
from gr1d_viz.eos import EOS

# Fill these paths to enable the final example.
EOS_FILES = {}  # e.g. {"thermo": "/path/to/eos.h5", "fractions": "/path/to/quark_fractions.h5"}
QUARK_SOURCE = None  # e.g. "fractions"; None resolves a uniquely available variable
EOS_TIME = None      # None = last density snapshot; otherwise simulation seconds

if EOS_FILES:
    eos = EOS(EOS_FILES)
    display(eos.available())
    quarks = eos.profile(run, "quark_fraction", source=QUARK_SOURCE)
    run.add_profile("quark_fraction", quarks)
    fig, axes = profiles(run, [
        Panel("rho", yscale="log"),
        Panel("quark_fraction", ylabel="Quark fraction (Xu + Xd + Xs)",
              ylim=(0, 1), twin=True),
    ], times=None if EOS_TIME is None else [EOS_TIME], xlim=(0, 100),
       title="Density and EOS-derived quark fraction versus radius")
    plt.show()
else:
    print("Set EOS_FILES to enable the density/quark-fraction radial plot.")
